# TP0 — Prise en main de Gurobi

Projet SDP — Côme-Alexis Puech, Hugo Vigna, Salomé Fonvielle

On résout les quatre programmes de la partie « À vous de jouer » avec les trois méthodes du notebook :
1. **en ligne** : une variable et une contrainte à la fois (`addVar`, `addConstr`) ;
2. **matricielle** : un vecteur de variables et `A @ x <= b` (`addMVar`) ;
3. **fichier `.lp`** : le programme est écrit dans un fichier, puis lu avec `read`.

In [1]:
import numpy as np
from gurobipy import Model, GRB, read

def afficher(m, noms):
    """Affiche la solution optimale d'un modèle résolu, ou signale qu'il n'y en a pas."""
    if m.SolCount == 0:  # statut INFEASIBLE (ou INF_OR_UNBD selon le presolve)
        print("Programme infaisable : aucune solution ne vérifie toutes les contraintes.")
        return
    valeurs = ", ".join(f"{n} = {v.X:g}" for n, v in zip(noms, m.getVars()))
    print(f"{valeurs}   |   objectif = {m.ObjVal:g}")

## $(P_1)$

Maximiser $x_1 + x_2$ s.c. $2x_1 + x_2 \le 14$, $\ -x_1 + 2x_2 \le 8$, $\ 2x_1 - x_2 \le 10$, $\ x_1, x_2 \ge 0$.

In [2]:
# Méthode 1 : en ligne
m = Model("P1")
m.Params.OutputFlag = 0
x1 = m.addVar(lb=0, name="x1")
x2 = m.addVar(lb=0, name="x2")
m.addConstr(2*x1 + x2 <= 14)
m.addConstr(-x1 + 2*x2 <= 8)
m.addConstr(2*x1 - x2 <= 10)
m.setObjective(x1 + x2, GRB.MAXIMIZE)
m.optimize()
afficher(m, ["x1", "x2"])

Set parameter Username


Set parameter LicenseID to value 2870724


Academic license - for non-commercial use only - expires 2027-10-02


x1 = 4, x2 = 6   |   objectif = 10


In [3]:
# Méthode 2 : matricielle
A = np.array([[2, 1], [-1, 2], [2, -1]])
b = np.array([14, 8, 10])
c = np.array([1, 1])

m = Model("P1 matriciel")
m.Params.OutputFlag = 0
x = m.addMVar(shape=2, lb=0)
m.addConstr(A @ x <= b)
m.setObjective(c @ x, GRB.MAXIMIZE)
m.optimize()
afficher(m, ["x1", "x2"])

x1 = 4, x2 = 6   |   objectif = 10


In [4]:
# Méthode 3 : fichier .lp
with open("P1.lp", "w") as f:
    f.write("""Maximize
 obj: x1 + x2
Subject To
 c1: 2 x1 + x2 <= 14
 c2: - x1 + 2 x2 <= 8
 c3: 2 x1 - x2 <= 10
End
""")  # les variables sont positives par défaut dans le format LP

m = read("P1.lp")
m.Params.OutputFlag = 0
m.optimize()
afficher(m, ["x1", "x2"])

Read LP format model from file P1.lp


Reading time = 0.00 seconds


obj: 3 rows, 2 columns, 6 nonzeros


x1 = 4, x2 = 6   |   objectif = 10


## $(P_2)$

Minimiser $14x_1 + 8x_2 + 10x_3$ s.c. $2x_1 - x_2 + 2x_3 \ge 1$, $\ x_1 + 2x_2 - x_3 \ge 1$, $\ x_1, x_2, x_3 \ge 0$.

*Remarque :* l'énoncé écrit « $2x_1 - x_2 + 2 \ge 1$ », sans doute une coquille pour $+\,2x_3$. Avec cette lecture,
$(P_2)$ est exactement le **dual** de $(P_1)$ : ses coefficients d'objectif $(14, 8, 10)$ sont les seconds membres de $(P_1)$,
et ses contraintes sont les colonnes de la matrice de $(P_1)$. On doit donc retrouver le même optimum, 10.

In [5]:
# Méthode 1 : en ligne
m = Model("P2")
m.Params.OutputFlag = 0
x1 = m.addVar(lb=0, name="x1")
x2 = m.addVar(lb=0, name="x2")
x3 = m.addVar(lb=0, name="x3")
m.addConstr(2*x1 - x2 + 2*x3 >= 1)
m.addConstr(x1 + 2*x2 - x3 >= 1)
m.setObjective(14*x1 + 8*x2 + 10*x3, GRB.MINIMIZE)
m.optimize()
afficher(m, ["x1", "x2", "x3"])

x1 = 0.6, x2 = 0.2, x3 = 0   |   objectif = 10


In [6]:
# Méthode 2 : matricielle
A = np.array([[2, -1, 2], [1, 2, -1]])
b = np.array([1, 1])
c = np.array([14, 8, 10])

m = Model("P2 matriciel")
m.Params.OutputFlag = 0
x = m.addMVar(shape=3, lb=0)
m.addConstr(A @ x >= b)
m.setObjective(c @ x, GRB.MINIMIZE)
m.optimize()
afficher(m, ["x1", "x2", "x3"])

x1 = 0.6, x2 = 0.2, x3 = 0   |   objectif = 10


In [7]:
# Méthode 3 : fichier .lp
with open("P2.lp", "w") as f:
    f.write("""Minimize
 obj: 14 x1 + 8 x2 + 10 x3
Subject To
 c1: 2 x1 - x2 + 2 x3 >= 1
 c2: x1 + 2 x2 - x3 >= 1
End
""")

m = read("P2.lp")
m.Params.OutputFlag = 0
m.optimize()
afficher(m, ["x1", "x2", "x3"])

Read LP format model from file P2.lp


Reading time = 0.00 seconds


obj: 2 rows, 3 columns, 6 nonzeros


x1 = 0.6, x2 = 0.2, x3 = 0   |   objectif = 10


## $(P_3)$

Maximiser $160A + 300B + 500C$ s.c. $3A + 3.5B + 5C \le 1800$, $\ 4A + 5B + 8C \le 2400$, $\ A + 1.5B + 8C \le 720$,
$\ A \ge 100$, $B \ge 150$, $C \ge 75$, $\ A, B, C \in \mathbb{N}$.

Les variables sont **entières** : c'est un programme linéaire en nombres entiers. Les bornes inférieures
se mettent directement dans `lb`.

*Remarque :* tel qu'il est écrit, $(P_3)$ est **infaisable**. Avec $A \\ge 100$, $B \\ge 150$, $C \\ge 75$, la troisième
contrainte donne $A + 1.5B + 8C \\ge 100 + 225 + 600 = 925 > 720$. Gurobi le détecte et renvoie le statut `INFEASIBLE` ;
on demande ensuite à Gurobi quelles contraintes sont en conflit (`computeIIS`).

In [8]:
# Méthode 1 : en ligne
m = Model("P3")
m.Params.OutputFlag = 0
A_ = m.addVar(lb=100, vtype=GRB.INTEGER, name="A")
B_ = m.addVar(lb=150, vtype=GRB.INTEGER, name="B")
C_ = m.addVar(lb=75, vtype=GRB.INTEGER, name="C")
m.addConstr(3*A_ + 3.5*B_ + 5*C_ <= 1800, name="c1")
m.addConstr(4*A_ + 5*B_ + 8*C_ <= 2400, name="c2")
m.addConstr(A_ + 1.5*B_ + 8*C_ <= 720, name="c3")
m.setObjective(160*A_ + 300*B_ + 500*C_, GRB.MAXIMIZE)
m.optimize()
afficher(m, ["A", "B", "C"])

# Sous-ensemble minimal de contraintes incompatibles (IIS)
m.computeIIS()
print("Contraintes en conflit :", [c.ConstrName for c in m.getConstrs() if c.IISConstr])
print("Bornes en conflit      :", [v.VarName for v in m.getVars() if v.IISLB])

Programme infaisable : aucune solution ne vérifie toutes les contraintes.
Contraintes en conflit : ['c3']
Bornes en conflit      : ['A', 'B', 'C']


In [9]:
# Méthode 2 : matricielle
M = np.array([[3, 3.5, 5], [4, 5, 8], [1, 1.5, 8]])
b = np.array([1800, 2400, 720])
c = np.array([160, 300, 500])

m = Model("P3 matriciel")
m.Params.OutputFlag = 0
x = m.addMVar(shape=3, lb=np.array([100, 150, 75]), vtype=GRB.INTEGER)
m.addConstr(M @ x <= b)
m.setObjective(c @ x, GRB.MAXIMIZE)
m.optimize()
afficher(m, ["A", "B", "C"])

Programme infaisable : aucune solution ne vérifie toutes les contraintes.


In [10]:
# Méthode 3 : fichier .lp
with open("P3.lp", "w") as f:
    f.write("""Maximize
 obj: 160 A + 300 B + 500 C
Subject To
 c1: 3 A + 3.5 B + 5 C <= 1800
 c2: 4 A + 5 B + 8 C <= 2400
 c3: A + 1.5 B + 8 C <= 720
Bounds
 A >= 100
 B >= 150
 C >= 75
General
 A B C
End
""")  # la section General déclare les variables entières

m = read("P3.lp")
m.Params.OutputFlag = 0
m.optimize()
afficher(m, ["A", "B", "C"])

Read LP format model from file P3.lp


Reading time = 0.00 seconds


obj: 3 rows, 3 columns, 9 nonzeros


Programme infaisable : aucune solution ne vérifie toutes les contraintes.


## $(P_4)$

Maximiser $c^T x$ s.c. $Ax \le b$, $x \ge 0$, avec
$A = \begin{pmatrix} 1 & 1\\ 1 & 3\\ 0 & 3 \end{pmatrix}$, $b = \begin{pmatrix} 150\\ 210\\ 180 \end{pmatrix}$, $c = \begin{pmatrix} 300\\ 500 \end{pmatrix}$.

In [11]:
# Méthode 1 : en ligne
m = Model("P4")
m.Params.OutputFlag = 0
x1 = m.addVar(lb=0, name="x1")
x2 = m.addVar(lb=0, name="x2")
m.addConstr(x1 + x2 <= 150)
m.addConstr(x1 + 3*x2 <= 210)
m.addConstr(3*x2 <= 180)
m.setObjective(300*x1 + 500*x2, GRB.MAXIMIZE)
m.optimize()
afficher(m, ["x1", "x2"])

x1 = 120, x2 = 30   |   objectif = 51000


In [12]:
# Méthode 2 : matricielle (forme naturelle pour P4)
A = np.array([[1, 1], [1, 3], [0, 3]])
b = np.array([150, 210, 180])
c = np.array([300, 500])

m = Model("P4 matriciel")
m.Params.OutputFlag = 0
x = m.addMVar(shape=2, lb=0)
m.addConstr(A @ x <= b)
m.setObjective(c @ x, GRB.MAXIMIZE)
m.optimize()
afficher(m, ["x1", "x2"])

x1 = 120, x2 = 30   |   objectif = 51000


In [13]:
# Méthode 3 : fichier .lp
with open("P4.lp", "w") as f:
    f.write("""Maximize
 obj: 300 x1 + 500 x2
Subject To
 c1: x1 + x2 <= 150
 c2: x1 + 3 x2 <= 210
 c3: 3 x2 <= 180
End
""")

m = read("P4.lp")
m.Params.OutputFlag = 0
m.optimize()
afficher(m, ["x1", "x2"])

Read LP format model from file P4.lp


Reading time = 0.00 seconds


obj: 3 rows, 2 columns, 5 nonzeros


x1 = 120, x2 = 30   |   objectif = 51000


## Bilan

Les trois méthodes donnent la même solution pour chaque programme ($(P_3)$ étant infaisable avec les trois).
On retrouve aussi la dualité : $(P_1)$ et $(P_2)$ ont le même optimum. La méthode en ligne est la plus lisible
pour un petit modèle ; la méthode matricielle est la plus adaptée quand les données sont déjà sous forme de
tableaux (comme $(P_4)$) ; le fichier `.lp` permet d'échanger un modèle indépendamment du code.
Dans le projet (`src/mrsort.py`), nous utilisons la méthode en ligne, avec en plus des variables binaires.